# Grammar of graphics with plotnine
Run from this folder, with data/ beside the notebook. Classroom examples and worked answers.


## The data for the first figure
Open 04_grammar.ipynb. One row is one synthetic household. Money is EUR per month.

In [ ]:
import plotnine as pn
pn.theme_set(pn.theme_gray(base_size=16))
from lesson_data import load_households
df = load_households()
columns = ["income", "saving", "region"]
print(df[columns].head(3).to_string(index=False))
print("Households:", len(df))

## Data, mapping, and a point layer
ggplot supplies the data. aes maps columns to x and y. geom_point draws the rows.

In [ ]:
p = (
    pn.ggplot(
        df,
        pn.aes("income", "saving")
    )
    + pn.geom_point()
)
p

## Changing the mapping swaps the axes
The observations stay the same. The positions change.

In [ ]:
p = (
    pn.ggplot(
        df, pn.aes(
            x="saving", y="income"
        )
    )
    + pn.geom_point()
)
p

## A constant colour belongs outside aes
Every point uses the same blue. alpha controls opacity.

In [ ]:
base = pn.ggplot(
    df, pn.aes("income", "saving")
)
p = base + pn.geom_point(
    color="#245b78", alpha=0.55
)
p

## A variable colour belongs inside aes
plotnine chooses a colour for each region and creates a legend.

In [ ]:
p = (
    pn.ggplot(
        df, pn.aes(
            "income", "saving",
            color="region"
        )
    )
    + pn.geom_point(alpha=0.6)
)
p

## A numeric colour uses a gradient
size is numeric. The colour scale now shows household size along a gradient.

In [ ]:
p = (
    pn.ggplot(
        df, pn.aes(
            "income", "saving",
            color="size"
        )
    )
    + pn.geom_point(alpha=0.7)
)
p

## A second layer marks zero saving
The horizontal line marks households whose income equals consumption.

In [ ]:
p = (
    base
    + pn.geom_hline(
        yintercept=0, color="#777777"
    )
    + pn.geom_point(alpha=0.55)
)
p

## Exercise 1: two-person households
Create the figure in the notebook. Explain why the colour legend appears.

In [ ]:
pairs = df.loc[df["size"] == 2].copy()
print("Households to plot:", len(pairs))
# Use pairs as the data.
# Map income to x and saving to y.
# Map region to colour.
# Add points with alpha=0.6 and a line at y=0.

## Exercise 1: worked answer
Filtering the data changes which households enter the figure.

In [ ]:
p = (
    pn.ggplot(
        pairs, pn.aes(
            "income", "saving",
            color="region"
        )
    )
    + pn.geom_hline(yintercept=0)
    + pn.geom_point(alpha=0.6)
)
p

## A histogram counts observations in bins
Each bar counts households in a 250 EUR saving interval.

In [ ]:
p = (
    pn.ggplot(df, pn.aes("saving"))
    + pn.geom_histogram(
        binwidth=250,
        fill="#245b78", color="white"
    )
)
p

## Bin width changes the visible pattern
The same 195 households now enter wider, 1000 EUR intervals.

In [ ]:
p = (
    pn.ggplot(df, pn.aes("saving"))
    + pn.geom_histogram(
        binwidth=1000,
        fill="#245b78", color="white"
    )
)
p

## A bar chart can count raw rows
geom_bar counts rows for each region.

In [ ]:
p = (
    pn.ggplot(df, pn.aes("region"))
    + pn.geom_bar(fill="#245b78")
)
p

## A table can already contain the heights
The next figure should show mean saving in EUR, using the rightmost column.

In [ ]:
summary = df.groupby("region", as_index=False).agg(
    households=("household_id", "size"),
    mean_saving=("saving", "mean"),
)
print(summary.round(1).to_string(index=False))

## geom_bar still counts summary rows
These bars are all 1: there is one summary row per region.

In [ ]:
p = (
    pn.ggplot(
        summary, pn.aes("region")
    )
    + pn.geom_bar(fill="#245b78")
)
p

## geom_col uses supplied heights
The y mapping supplies the bar heights directly.

In [ ]:
p = (
    pn.ggplot(
        summary, pn.aes(
            "region", "mean_saving"
        )
    )
    + pn.geom_col(fill="#245b78")
)
p

## A boxplot compares distributions
The line in each box marks the median saving. The box covers the middle half.

In [ ]:
p = (
    pn.ggplot(
        df,
        pn.aes("region", "saving")
    )
    + pn.geom_boxplot(fill="#dbe7ed")
)
p

## The quartiles behind the boxes
The 25th and 75th percentiles form the box. The 50th percentile is the median.

In [ ]:
quartiles = (
    df.groupby("region")["saving"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
)
print(quartiles.to_string())

## Facets repeat a plot for each region
Each panel shows one region. The default axes use common scales.

In [ ]:
p = (
    pn.ggplot(
        df,
        pn.aes("income", "saving")
    )
    + pn.geom_point(alpha=0.6)
    + pn.facet_wrap(
        "~region", ncol=2
    )
)
p

## The same facet works with a histogram
Every panel uses the same 250 EUR bin width and the same axis scales.

In [ ]:
p = (
    pn.ggplot(df, pn.aes("saving"))
    + pn.geom_histogram(
        binwidth=250,
        fill="#245b78", color="white"
    )
    + pn.facet_wrap(
        "~region", ncol=2
    )
)
p

## Exercise 2: income per person
Use the existing column from Lecture 8. Identify the x and computed y variables.

In [ ]:
print(df["income_per_person"].head(3))
# Plot a histogram of income_per_person.
# Use a bin width of 200 EUR.
# Draw one panel per region, in two columns.
# Keep the same bin width and axes across panels.

## Exercise 2: worked answer
The data transformation happened in pandas. The histogram counts the transformed values.

In [ ]:
mapping = pn.aes(
    "income_per_person")
p = pn.ggplot(df, mapping)
p = p + pn.geom_histogram(
    binwidth=200,
    fill="#245b78", color="white")
p = p + pn.facet_wrap(
    "~region", ncol=2)
p

## Saving shares as percentages
Two zero-income households have undefined saving shares. This figure uses the other 193.

In [ ]:
rates = df.dropna(subset=["saving_share"]).copy()
rates["saving_pct"] = 100 * rates["saving_share"]
print("Plotted households:", len(rates))
print(rates[["saving_share", "saving_pct"]].head(3).round(2)
      .to_string(index=False))

## A new mapping uses the new units
The y values and the axis label now agree about percentages.

In [ ]:
mapping = pn.aes(
    "income_per_person",
    "saving_pct", color="region")
p = pn.ggplot(rates, mapping)
p = p + pn.geom_point(alpha=0.6)
p = p + pn.labs(
    y="Saving (% of income)")
p

## A categorical household size
Converting size to a category creates one distribution per household-size group.

In [ ]:
df["size_group"] = (
    df["size"].astype("category")
)
p = (
    pn.ggplot(
        df, pn.aes(
            "size_group", "saving"
        )
    )
    + pn.geom_boxplot()
)
p

## Mappings can belong to one layer
The point layer maps colour to region. The zero-saving line keeps a constant appearance.

In [ ]:
p = (
    pn.ggplot(
        df,
        pn.aes("income", "saving")
    )
    + pn.geom_hline(yintercept=0)
    + pn.geom_point(
        pn.aes(color="region"),
        alpha=0.6
    )
)
p